In [ ]:
"""
Statistical Significance Testing Script.

Balanced VS. Balanced(+1646)
"""

import numpy as np
import pandas as pd
from scipy import stats

DATA = {

    #  5-CLASS TASK (Balanced VS. Balanced(+1646))

    ("5-Class", "Accuracy"): {
        "Baseline":  [72.86, 73.03, 69.94, 70.99, 68.95, 70.53, 75.61, 72.37, 72.58],
        "Augmented": [72.53, 72.39, 70.32, 71.54, 68.75, 72.48, 76.79, 73.76, 72.62]
    },
    ("5-Class", "Weighted-F1"): {
        "Baseline":  [73.14, 73.27, 70.32, 71.33, 69.22, 70.28, 75.76, 72.55, 72.85],
        "Augmented": [72.87, 72.61, 70.70, 72.01, 69.14, 72.32, 76.93, 73.98, 72.88]
    },
    ("5-Class", "Macro-F1"): {
        "Baseline":  [63.95, 62.45, 58.25, 62.12, 55.71, 55.95, 71.03, 66.23, 66.17],
        "Augmented": [65.90, 60.49, 60.81, 62.79, 58.41, 58.51, 73.31, 67.44, 66.15]
    },
    ("Binary", "Accuracy"): {
        "Baseline":  [93.58, 94.36, 91.21, 91.75, 89.20, 94.54, 96.76, 94.27, 93.70],
        "Augmented": [93.73, 94.21, 91.87, 92.11, 91.12, 94.60, 96.31, 94.72, 94.93]
    },
    ("Binary", "Weighted-F1"): {
        "Baseline":  [93.72, 94.43, 91.50, 91.95, 89.50, 94.58, 96.77, 94.33, 93.81],
        "Augmented": [93.81, 94.26, 92.04, 92.29, 91.33, 94.64, 96.34, 94.76, 94.99]
    },
    ("Binary", "Macro-F1"): {
        "Baseline":  [89.44, 90.53, 85.89, 86.50, 82.49, 90.75, 94.45, 90.36, 89.55],
        "Augmented": [89.49, 90.22, 86.61, 87.04, 85.47, 90.85, 93.76, 91.07, 91.49]
    }
}

def holm_bonferroni(p_values):
    indexed = sorted(enumerate(p_values), key=lambda x: x[1])    #Enforces monotonicity in Holm-Bonferroni step-down correction.
    m = len(p_values)
    adj = [0.0] * m
    prev = 0.0
    for rank, (original_idx, p) in enumerate(indexed):
        val = min(p * (m - rank), 1.0)
        val = max(val, prev)  # enforce monotonicity
        adj[original_idx] = val
        prev = val
    return adj

records = []
raw_wilcox_p = []
raw_ttest_p = []

for (task, metric), values in DATA.items():
    base = np.array(values["Baseline"], dtype=float)
    aug = np.array(values["Augmented"], dtype=float)
    diff = aug - base

    d_nonzero = diff[diff != 0]
    n_nonzero = len(d_nonzero)
    n_total = len(diff)

    mean_base = np.mean(base)
    mean_aug = np.mean(aug)
    mean_gain = np.mean(diff)
    wins = int(np.sum(diff > 0))

    # 1. Shapiro-Wilk test on differences
    shapiro_stat, p_shapiro = stats.shapiro(diff)
    is_normal = "Yes" if p_shapiro >= 0.05 else "No"

    # 2. Statistical Tests
    t_stat, p_ttest = stats.ttest_rel(aug, base)
    w_res = stats.wilcoxon(aug, base)
    p_wilcox = w_res.pvalue

    raw_wilcox_p.append(p_wilcox)
    raw_ttest_p.append(p_ttest)

    # 3. Correct Wilcoxon Z & Effect Size r
    ranks = stats.rankdata(np.abs(d_nonzero))
    w_plus = ranks[d_nonzero > 0].sum()
    w_minus = ranks[d_nonzero < 0].sum()

    mean_w = n_nonzero * (n_nonzero + 1) / 4.0
    std_w = np.sqrt(n_nonzero * (n_nonzero + 1) * (2 * n_nonzero + 1) / 24.0)
    z_val = (w_plus - mean_w) / std_w
    effect_r = abs(z_val) / np.sqrt(n_nonzero)

    # 4. Cohen's d for paired samples
    cohen_d = mean_gain / np.std(diff, ddof=1)

    records.append({
        "Task": task,
        "Metric": metric,
        "Mean_Base": round(mean_base, 2),
        "Mean_Aug": round(mean_aug, 2),
        "Gain": round(mean_gain, 2),
        "Wins": f"{wins}/{n_total}",
        "Shapiro_p": round(p_shapiro, 4),
        "Is_Normal": is_normal,
        "Wilcoxon_p": round(p_wilcox, 4),
        "ttest_p": round(p_ttest, 4),
        "Z_val": round(z_val, 3),
        "Effect_r": round(effect_r, 3),
        "Cohens_d": round(cohen_d, 2)
    })

# Apply Holm to both sets of p-values
adj_p_wilcox = holm_bonferroni(raw_wilcox_p)
adj_p_ttest = holm_bonferroni(raw_ttest_p)

for i, rec in enumerate(records):
    rec["Holm_Wilcoxon_p"] = round(adj_p_wilcox[i], 4)
    rec["Holm_ttest_p"] = round(adj_p_ttest[i], 4)
    rec["Sig_Wilcoxon"] = "Yes" if adj_p_wilcox[i] < 0.05 else "No"
    rec["Sig_ttest"] = "Yes" if adj_p_ttest[i] < 0.05 else "No"

df_final = pd.DataFrame(records)
print(df_final.to_string(index=False))



output_file = "comprehensive_significance_report_BvsB1646.csv"
df_final.to_csv(output_file, index=False)


try:
    from google.colab import files
    files.download(output_file)
except ImportError:
    print(f"Saved locally'{output_file}'")